# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## 1. Objetivo y decisiones de diseño
Construir dos rankings léxicos independientes —TF-IDF con coseno y BM25— sobre
documentos originales completos de NFCorpus, una colección de nutrición y salud en
inglés. Se concatena **título + espacio + texto**: el título aporta el tema y el
contenido aporta evidencia y terminología. Cada documento conserva su `_id`, sin
fragmentación ni ponderación adicional del título. Los fragmentos de la API son
solo una presentación de los primeros 300 caracteres del texto original.

Fuentes oficiales: [documentos y consultas](https://huggingface.co/datasets/BeIR/nfcorpus)
y [qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels). Se reutilizan los
archivos locales `nfcorpus/nfcorpus`; no se necesita descargar datos para ejecutar.
Los qrels de `train` y `dev` quedan fuera de esta evaluación. No se ajustan
parámetros con `test`. Los módulos Python son la única implementación de los
modelos y también los importa la API, para evitar diferencias entre ambas vías.

## 2. Dependencias e importaciones
Desde la carpeta raíz, instalar una vez: `python -m pip install -r requirements-dev.txt`.
Seleccionar ese entorno como kernel del notebook y ejecutar todas las celdas.
Las versiones fijadas están en los archivos de dependencias. NLTK se usa solo para
Snowball; no se descargan modelos lingüísticos, tokenizadores ni listas externas.
La siguiente celda localiza el proyecto tanto desde su raíz como desde una subcarpeta.

In [1]:
from pathlib import Path
import sys, json, math, inspect, subprocess, importlib.metadata
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'retrieval.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook dentro de la carpeta del proyecto.')
sys.path.insert(0, str(ROOT))
from preprocessing import normalize, tokenize, preprocess, STOPWORDS, stem
from retrieval import SearchIndex
from evaluation import load_dataset, evaluate, metrics_at_k
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0])
display(pd.DataFrame([(p, importlib.metadata.version(p)) for p in
    ['numpy','scipy','pandas','nltk','fastapi','uvicorn','pydantic','httpx']],
    columns=['Paquete','Versión']))

Python: 3.13.14


,Paquete,Versión
0,numpy,2.2.5
1,scipy,1.15.2
2,pandas,2.3.1
3,nltk,3.10.4
4,fastapi,0.164.0
5,uvicorn,0.54.0
6,pydantic,2.12.5
7,httpx,0.28.1


## 3. Carga y exploración del corpus
Se validan JSON/TSV, tipos, IDs únicos, referencias de todos los qrels y grados
no negativos. Los conteos conocidos permiten detectar una descarga incompleta.
Los identificadores se conservan como cadenas de texto.

In [2]:
documents, queries, qrels = load_dataset()
report = {'documents': len(documents), 'queries': len(queries), 'test_queries': len(qrels),
          'test_judgments': sum(map(len, qrels.values())),
          'grades': sorted({g for judgments in qrels.values() for g in judgments.values()})}
assert report['documents'] == 3633
assert report['queries'] == 3237
assert report['test_queries'] == 323
assert report['test_judgments'] == 12334
display(pd.DataFrame([(k, v) for k, v in report.items() if k != 'files'], columns=['Dato','Valor']))
print('Estructura de documento:', list(documents[0]))
display(pd.DataFrame(documents[:2])[['_id','title','text']])
display(pd.DataFrame([{'query_id': qid, 'query': queries[qid],
                       'judgments': len(qrels[qid])} for qid in sorted(qrels)[:4]]))
first_qid = sorted(qrels)[0]
print('Qrels de ejemplo:', first_qid, list(qrels[first_qid].items())[:5])

,Dato,Valor
0,documents,3633
1,queries,3237
2,test_queries,323
3,test_judgments,12334
4,grades,"[1, 2]"


Estructura de documento: ['_id', 'title', 'text', 'metadata']


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...


,query_id,query,judgments
0,PLAIN-1008,deafness,5
1,PLAIN-1018,DHA,60
2,PLAIN-102,Stopping Heart Disease in Childhood,24
3,PLAIN-1028,dietary scoring,1


Qrels de ejemplo: PLAIN-1008 [('MED-4532', 1), ('MED-4533', 1), ('MED-4534', 1), ('MED-4535', 1), ('MED-4536', 1)]


## 4. Preprocesamiento textual
1. **NFKC Unicode:** unifica variantes de compatibilidad (por ejemplo, letras de
   ancho completo). El texto original se almacena sin cambios.
2. **Minúsculas:** une variantes de capitalización.
3. **Tokenización:** conserva números y decimales, letras griegas, identificadores
   como `B12` y símbolos como `%`, `>=`, `±`. Se separan compuestos con guion.
   Se expande `n't` a `not`. No se aplica un filtro que suprima todos los números.
4. **Stopwords:** lista inglesa explícita en `preprocessing.py`; se conservan
   `no`, `nor`, `not`, `without`, `never` por su posible significado clínico.
5. **Stemming Snowball inglés:** agrupa variantes morfológicas sin descargar
   diccionarios. Es reproducible y sencillo de explicar; puede confluir términos
   distintos y producir raíces que no son palabras. La lematización requeriría
   recursos y decisiones gramaticales adicionales.

Se aplica exactamente la misma función a consultas y documentos. Conservar una
negación o un símbolo no convierte al modelo de bolsa de palabras en un sistema
capaz de interpretar negación, relaciones químicas o afirmaciones médicas.
La puntuación no incluida en la regla se considera separador; no se aplica
expansión semántica ni detección de unidades.

In [3]:
examples = [documents[0]['title'] + ' ' + documents[0]['text'][:230],
    "Vitamin B12: 2.5 mg; NOT without β-carotene, risk >= 5%. Isn't harmful.",
    queries[first_qid]]
display(pd.DataFrame([{'Original': text, 'Unicode/minúsculas': normalize(text),
    'Tokens': tokenize(text), 'Sin stopwords': [t for t in tokenize(text) if t not in STOPWORDS],
    'Stemming': preprocess(text)} for text in examples]))
print(inspect.getsource(preprocess))

,Original,Unicode/minúsculas,Tokens,Sin stopwords,Stemming
0,Statin Use and Breast Cancer Survival: A Natio...,statin use and breast cancer survival: a natio...,"[statin, use, and, breast, cancer, survival, a...","[statin, use, breast, cancer, survival, nation...","[statin, use, breast, cancer, surviv, nationwi..."
1,"Vitamin B12: 2.5 mg; NOT without β-carotene, r...","vitamin b12: 2.5 mg; not without β-carotene, r...","[vitamin, b12, 2.5, mg, not, without, β, carot...","[vitamin, b12, 2.5, mg, not, without, β, carot...","[vitamin, b12, 2.5, mg, not, without, β, carot..."
2,deafness,deafness,[deafness],[deafness],[deaf]


def preprocess(text: str) -> list[str]:
    return [stem(t) for t in tokenize(text) if t not in STOPWORDS]



## 5. Vocabulario e índice invertido
El vocabulario asigna una columna interna a cada término. Para cada término,
`postings[t]` almacena pares `(posición interna, TF)`; `doc_ids[posición]` recupera
el ID original. La matriz dispersa CSR conserva las frecuencias y CSC permite
acceder a los pesos por término. DF cuenta documentos, no ocurrencias.
La longitud es el número de tokens **después** del procesamiento y `avgdl` es
su media sobre los N documentos, incluidos los vacíos si existieran.

Los candidatos son la **unión de postings de los términos conocidos de la
consulta**. Se puntúan esos documentos y se ordenan por score descendente; ante
empates se utiliza el ID original en orden lexicográfico. Si hay menos candidatos
que `k`, se devuelve esa cantidad; sin coincidencias se devuelve `[]`.
No se completan rankings con documentos arbitrarios de puntuación cero.

In [4]:
index = SearchIndex(documents)
index.save(ARTIFACTS / 'index')
display(pd.DataFrame([{'N': index.N, 'Vocabulario': len(index.vocabulary),
    'Pares término/documento': index.tf.nnz, 'avgdl': index.avgdl,
    'Longitud mínima': index.doc_lengths.min(), 'Longitud máxima': index.doc_lengths.max()}]))
display(pd.DataFrame([{'term': term, 'column': col, 'DF': int(index.df[col])}
    for term, col in list(index.vocabulary.items())[:12]]))
sample_term = preprocess('cholesterol')[0]
display(pd.DataFrame([{'term': sample_term, 'position': int(pos),
    'document_id': index.doc_ids[pos], 'TF': int(tf), 'length': int(index.doc_lengths[pos])}
    for pos, tf in index.postings[sample_term][:8]]))
assert index.doc_ids == [d['_id'] for d in documents]
assert np.all(index.doc_lengths == index.tf.sum(axis=1).A1)
print(inspect.getsource(SearchIndex.score))

,N,Vocabulario,Pares término/documento,avgdl,Longitud mínima,Longitud máxima
0,3633,22797,365019,166.370493,11,1063


,term,column,DF
0,%,0,1580
1,+,1,312
2,-0,2,1
3,-0.0,3,1
4,-0.0.98,4,1
5,-0.00,5,2
6,-0.002,6,1
7,-0.003,7,1
8,-0.00384,8,1
9,-0.004,9,1


,term,position,document_id,TF,length
0,cholesterol,20,MED-711,4,228
1,cholesterol,21,MED-712,1,184
2,cholesterol,33,MED-726,3,177
3,cholesterol,50,MED-751,7,208
4,cholesterol,53,MED-754,4,351
5,cholesterol,61,MED-816,2,168
6,cholesterol,67,MED-823,1,201
7,cholesterol,79,MED-835,1,207


    def score(self, query, model):
        if model not in {'tfidf', 'bm25'}:
            raise ValueError('model debe ser tfidf o bm25')
        terms = sorted(set(preprocess(query)) & self.vocabulary.keys())
        scores = np.zeros(self.N)
        if not terms:
            return scores, np.array([], dtype=int)
        candidates = np.unique(np.concatenate([self.postings[t][:, 0] for t in terms]))

        if model == 'tfidf':
            vector = self.query_vector(query)
            norm = np.sqrt(vector.multiply(vector).sum())
            for col, query_weight in zip(vector.indices, vector.data):
                start, end = self.tfidf_csc.indptr[col:col + 2]
                positions = self.tfidf_csc.indices[start:end]
                scores[positions] += self.tfidf_csc.data[start:end] * query_weight
            denominator = self.doc_norms[candidates] * norm
            scores[candidates] = np.divide(scores[candidates], denominator,
                out=np.zeros_like(denominator

## 6. TF-IDF y similitud coseno
Para frecuencias positivas:

$$TF(t,d)=1+\ln f(t,d),\quad IDF(t)=1+\ln\frac{N+1}{df(t)+1}$$
$$w_{t,d}=TF(t,d)IDF(t),\qquad
\cos(q,d)=\frac{\sum_t w_{t,q}w_{t,d}}{\|q\|_2\|d\|_2}.$$

Los pesos ausentes valen cero. Las consultas usan el vocabulario e IDF del
corpus, y TF logarítmica para repeticiones. Los términos desconocidos se ignoran.
Las normas de los documentos se precalculan; la norma de cada consulta se calcula
en la búsqueda. Una norma nula produce un ranking vacío, sin dividir para cero.
El coseno normaliza por la norma del vector completo del documento.

**Comprobación manual:** D1=`apple apple banana`, D2=`banana carrot`,
q=`apple banana`. N=2, df(apple)=1, df(banana)=2;
IDF(apple)=1+ln(3/2), IDF(banana)=1. En D1, el peso de apple es
(1+ln 2)·IDF(apple). Se compara el cálculo escalar con la implementación.

In [5]:
tiny = SearchIndex([{'_id':'D1','title':'','text':'apple apple banana'},
                    {'_id':'D2','title':'','text':'banana carrot'}])
ia = 1 + math.log(3/2)
wa = (1 + math.log(2)) * ia
manual_cosine = (wa * ia + 1) / (math.sqrt(wa**2 + 1) * math.sqrt(ia**2 + 1))
actual_cosine = tiny.search('apple banana', 'tfidf', 2)
assert math.isclose(actual_cosine[0]['score'], manual_cosine, abs_tol=1e-12)
print('Coseno D1 manual:', manual_cosine)
display(pd.DataFrame(actual_cosine))
print('Vector disperso q:', tiny.query_vector('apple banana').toarray())

Coseno D1 manual: 0.9757694105051148


,rank,document_id,score,title,snippet
0,1,D1,0.975769,,apple apple banana
1,2,D2,0.336097,,banana carrot


Vector disperso q: [[1.40546511 1.         0.        ]]


## 7. BM25
$$IDF_{BM25}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$
$$BM25(q,d)=\sum_{t\in unique(q)} IDF_{BM25}(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}.$$

Se fijan **k1=1.2** y **b=0.75** antes de evaluar. Son valores iniciales
habituales y los solicitados para esta implementación; no se optimizan con test.
k1 controla cuánto tarda en saturarse la contribución de TF: repetir un término
no incrementa indefinidamente su peso. IDF favorece términos presentes en menos
documentos. b interpola entre omitir la normalización (b=0) y aplicarla totalmente
(b=1). Aquí la longitud se compara con la media del corpus.

A diferencia del coseno, BM25 no divide por la norma vectorial completa; normaliza
la TF de cada término según la longitud y la satura. La suma considera cada
término distinto de la consulta una vez, sin factor de frecuencia de consulta.
Cada modelo calcula sus propios scores, sin mezclar ni reutilizar el otro ranking.

En el mismo ejemplo pequeño, avgdl=2.5, IDF(apple)=ln 2 e
IDF(banana)=ln 1.2. Se verifica la suma de las dos contribuciones para D1.

In [6]:
manual_bm25 = math.log(2)*2*2.2/(2+1.2*(.25+.75*3/2.5))
manual_bm25 += math.log(1.2)*2.2/(1+1.2*(.25+.75*3/2.5))
actual_bm25 = tiny.search('apple banana','bm25',2)
assert math.isclose(actual_bm25[0]['score'],manual_bm25,abs_tol=1e-12)
print('BM25 D1 manual:', manual_bm25)
display(pd.DataFrame(actual_bm25))
display(pd.DataFrame(tiny.explain('apple banana','D1')))

BM25 D1 manual: 1.0708543050001982


,rank,document_id,score,title,snippet
0,1,D1,1.070854,,apple apple banana
1,2,D2,0.198568,,banana carrot


,term,tf,df,length,tfidf_contribution,bm25_contribution
0,appl,2,1,3,0.751172,0.902322
1,banana,1,2,3,0.224597,0.168533


## 8. Ejemplos de recuperación real
Se muestran una consulta oficial de prueba y otra escrita libremente. Los scores
de TF-IDF y BM25 tienen escalas diferentes; no deben compararse numéricamente entre
modelos. `top_k` se decide en cada solicitud. Los IDs mostrados provienen del corpus.

In [7]:
for text in [queries[first_qid], 'fermented chickpeas exercise inflammation 2026']:
    display(Markdown(f'**Consulta:** {text}'))
    for model in ['tfidf','bm25']:
        print(model)
        display(pd.DataFrame(index.search(text,model,5)))
print('Fuera de vocabulario:', index.search('zxqvprsnm','bm25',10))

**Consulta:** deafness

tfidf


,rank,document_id,score,title,snippet
0,1,MED-4668,0.151697,Cretinism revisited.,Endemic cretinism includes two syndromes: a mo...


bm25


,rank,document_id,score,title,snippet
0,1,MED-4668,9.046572,Cretinism revisited.,Endemic cretinism includes two syndromes: a mo...


**Consulta:** fermented chickpeas exercise inflammation 2026

tfidf


,rank,document_id,score,title,snippet
0,1,MED-2009,0.242176,Nutritional quality and health benefits of chi...,Chickpea (Cicer arietinum L.) is an important ...
1,2,MED-5204,0.168295,Relevance of protein fermentation to gut health.,It is generally accepted that carbohydrate fer...
2,3,MED-4125,0.152376,Human gut microbiota does not ferment erythritol.,"Erythritol, a naturally occurring polyol, is g..."
3,4,MED-3583,0.141238,First and second meal effects of pulses on blo...,Pulses are low-glycemic appetite-suppressing f...
4,5,MED-2298,0.137344,A guide to exercise prescription.,Exercise is a fundamental component of good he...


bm25


,rank,document_id,score,title,snippet
0,1,MED-2009,11.994529,Nutritional quality and health benefits of chi...,Chickpea (Cicer arietinum L.) is an important ...
1,2,MED-4269,9.986521,Butyrate and other short-chain fatty acids as ...,PURPOSE OF REVIEW: High-fiber diets have been ...
2,3,MED-3583,9.945481,First and second meal effects of pulses on blo...,Pulses are low-glycemic appetite-suppressing f...
3,4,MED-3450,9.338472,"Exercise-induced oxidative stress:myths, reali...",Although assays for the most popular markers o...
4,5,MED-3465,9.026385,Effect of blueberry ingestion on natural kille...,Blueberries are rich in antioxidants known as ...


Fuera de vocabulario: []


## 9. Evaluación con las 323 consultas test
Se usan **todas las consultas cuyos IDs aparecen en `qrels/test.tsv`**; sus IDs
se muestran abajo y los resultados individuales se exportan. Las 3.237 consultas
del archivo completo incluyen otros splits; no se evalúan mezclándolos.

Relevante binario para precisión/recall: grado > 0. Documento sin juicio: grado 0
en esta evaluación, lo que **no demuestra** que sea irrelevante en el mundo real.
Para NDCG se conservan los grados 1 y 2, con ganancia exponencial **2^grado−1**:

$$P@10=\frac{\#relevantes\ en\ top10}{10},\qquad
R@10=\frac{\#relevantes\ en\ top10}{\#relevantes\ de\ la\ consulta}$$
$$DCG@10=\sum_{i=1}^{10}\frac{2^{rel_i}-1}{\log_2(i+1)},\quad
NDCG@10=\frac{DCG@10}{IDCG@10}.$$

IDCG ordena los grados de **todos los juicios de la consulta**, no solo los
recuperados. Los rangos ausentes aportan cero; P@10 siempre divide para 10 aunque
el ranking sea más corto. Si no hay relevantes, Recall=0; si IDCG=0, NDCG=0.
El promedio es **macro**: cada consulta tiene el mismo peso.
La convención exponencial se declara porque una ganancia lineal daría otros
valores en una colección con relevancia graduada.

Los qrels se usan después de recuperar. La búsqueda solo recibe texto, modelo
y k. No se filtran ni se reordenan resultados según las etiquetas.

In [8]:
print('IDs de todas las consultas evaluadas:')
print(', '.join(sorted(qrels)))
print(inspect.getsource(metrics_at_k))
hand = metrics_at_k(['B','X','A'], {'A':2,'B':1,'C':1},3)
expected_ndcg = (1+3/math.log2(4))/(3+1/math.log2(3)+1/math.log2(4))
assert math.isclose(hand['NDCG@3'],expected_ndcg)
assert hand['Precision@3'] == 2/3 and hand['Recall@3'] == 2/3
assert metrics_at_k([], {}) == {'Precision@10':0.,'Recall@10':0.,'NDCG@10':0.}
print('Ejemplo manual de métricas:', hand)
per_query, summary, runs = evaluate(index, queries, qrels, 10)
per_query.to_csv(ARTIFACTS / 'metrics_per_query.csv', index=False)
summary.to_csv(ARTIFACTS / 'metrics_summary.csv')
display(summary.rename_axis('Modelo').rename(index={'tfidf':'TF-IDF','bm25':'BM25'}).style.format('{:.6f}'))
display(per_query.head(10))
print('Consultas por modelo:', per_query.groupby('model').size().to_dict())

IDs de todas las consultas evaluadas:
PLAIN-1008, PLAIN-1018, PLAIN-102, PLAIN-1028, PLAIN-1039, PLAIN-1050, PLAIN-1066, PLAIN-1088, PLAIN-1098, PLAIN-1109, PLAIN-1119, PLAIN-112, PLAIN-1130, PLAIN-1141, PLAIN-1151, PLAIN-1161, PLAIN-1172, PLAIN-1183, PLAIN-1193, PLAIN-12, PLAIN-1203, PLAIN-1214, PLAIN-1225, PLAIN-123, PLAIN-1236, PLAIN-1249, PLAIN-1262, PLAIN-1275, PLAIN-1288, PLAIN-1299, PLAIN-1309, PLAIN-1320, PLAIN-133, PLAIN-1331, PLAIN-1342, PLAIN-1353, PLAIN-1363, PLAIN-1374, PLAIN-1387, PLAIN-1398, PLAIN-1409, PLAIN-1419, PLAIN-1429, PLAIN-143, PLAIN-1441, PLAIN-1453, PLAIN-1463, PLAIN-1473, PLAIN-1485, PLAIN-1496, PLAIN-1506, PLAIN-1516, PLAIN-1527, PLAIN-153, PLAIN-1537, PLAIN-1547, PLAIN-1557, PLAIN-1568, PLAIN-1579, PLAIN-1590, PLAIN-1601, PLAIN-1611, PLAIN-1621, PLAIN-1635, PLAIN-1645, PLAIN-165, PLAIN-1656, PLAIN-1667, PLAIN-1679, PLAIN-1690, PLAIN-1700, PLAIN-1710, PLAIN-1721, PLAIN-1731, PLAIN-1741, PLAIN-175, PLAIN-1752, PLAIN-1762, PLAIN-1772, PLAIN-1784, PLAIN-1794, 

,Precision@10,Recall@10,NDCG@10
Modelo,,,
TF-IDF,0.232508,0.153092,0.323104
BM25,0.238080,0.152893,0.328409


,model,query_id,Precision@10,Recall@10,NDCG@10
0,tfidf,PLAIN-1008,0.0,0.000000,0.000000
1,tfidf,PLAIN-1018,0.6,0.100000,0.636474
2,tfidf,PLAIN-102,0.0,0.000000,0.000000
3,tfidf,PLAIN-1028,0.0,0.000000,0.000000
4,tfidf,PLAIN-1039,0.2,1.000000,1.000000
5,tfidf,PLAIN-1050,0.1,0.010638,0.063621
6,tfidf,PLAIN-1066,0.0,0.000000,0.000000
7,tfidf,PLAIN-1088,0.2,0.400000,0.553146
8,tfidf,PLAIN-1098,0.0,0.000000,0.000000
9,tfidf,PLAIN-1109,0.5,0.051020,0.620397


Consultas por modelo: {'bm25': 323, 'tfidf': 323}


## 10. Comparación e interpretación
Los casos se eligen **después** de evaluar para describir las diferencias de
NDCG@10, sin cambiar el preprocesamiento ni los parámetros. Esto es análisis del
test y no evidencia de generalización a otro corpus. Se muestran los diez
resultados, su grado y longitud, además de las contribuciones de cada término
en los primeros documentos.

En `PLAIN-3332` (*Alcohol Risks vs. Benefits*), TF-IDF coloca `MED-4994`
(grado 2) primero, mientras BM25 lo coloca tercero. En BM25, `MED-1680`, sin
juicio positivo para esta consulta, sube al primer puesto: sus frecuencias de
`alcohol`, `risk` y `vs` suman más contribución aunque su longitud es mayor.
La tabla de contribuciones permite verificar esta explicación directamente.

En `PLAIN-3342` (*Is Coconut Milk Good For You?*), BM25 coloca primero
`MED-5012` (grado 2) y segundo `MED-5014` (grado 2). TF-IDF favorece primero
`MED-5015`, un documento corto sobre uso intravenoso de agua de coco: su norma
vectorial y el término raro `coconut` producen un coseno alto, aunque no tiene
juicio positivo aquí. BM25 acumula coincidencias adicionales y TF saturada.

En `PLAIN-1008` (*deafness*), ambos solo recuperan `MED-4668`, sin juicio
positivo. La raíz `deaf` aparece en un solo documento y los cinco documentos
relevantes no comparten ese término procesado. Es una limitación de coincidencia
léxica: no se recuperan automáticamente sinónimos o conceptos relacionados.
Conservar negaciones tampoco permite comprender sus relaciones. Los juicios
incompletos también limitan la interpretación de resultados sin etiqueta.

In [9]:
comparison = per_query.pivot(index='query_id',columns='model',values='NDCG@10')
comparison['delta_tfidf_bm25'] = comparison['tfidf'] - comparison['bm25']
cases = {'TF-IDF superior': comparison['delta_tfidf_bm25'].idxmax(),
         'BM25 superior': comparison['delta_tfidf_bm25'].idxmin(),
         'Ambos limitados': comparison[['tfidf','bm25']].max(axis=1).idxmin()}
display(comparison.loc[list(cases.values())])
for label, qid in cases.items():
    display(Markdown(f'### {label}: {qid} — {queries[qid]}'))
    for model in ['tfidf','bm25']:
        enriched = [{**r, 'grade': qrels[qid].get(r['document_id'],0),
            'length': int(index.doc_lengths[index.id_to_position[r['document_id']]])}
            for r in runs[model][qid]]
        print(model)
        display(pd.DataFrame(enriched)[['rank','document_id','score','grade','length','title']])
        for r in runs[model][qid][:2]:
            print('Contribuciones:', r['document_id'])
            display(pd.DataFrame(index.explain(queries[qid],r['document_id'])))
        if len(runs[model][qid]) < 10:
            print('Menos de 10 candidatos; las posiciones restantes aportan cero.')

model,bm25,tfidf,delta_tfidf_bm25
query_id,,,
PLAIN-3332,0.439680,0.848619,0.408939
PLAIN-3342,0.765361,0.498189,-0.267171
PLAIN-1008,0.000000,0.000000,0.000000


### TF-IDF superior: PLAIN-3332 — Alcohol Risks vs. Benefits

tfidf


,rank,document_id,score,grade,length,title
0,1,MED-4994,0.191145,2,137,Who benefits most from the cardioprotective pr...
1,2,MED-3696,0.188357,1,175,Alcohol consumption and breast cancer risk in ...
2,3,MED-3762,0.186291,1,180,Moderate alcohol consumption during adult life...
3,4,MED-3768,0.179868,1,97,The positive and negative health effects of al...
4,5,MED-3766,0.174148,1,172,Epidemiology and pathophysiology of alcohol an...
5,6,MED-1680,0.170462,0,273,Effect of potentially modifiable risk factors ...
6,7,MED-4445,0.169096,0,178,Alcohol intake and mortality among women with ...
7,8,MED-3765,0.162777,1,63,Molecular mechanisms of alcohol-mediated carci...
8,9,MED-1678,0.160941,0,200,Combined effect of low-risk dietary and lifest...
9,10,MED-3763,0.150390,1,158,A single sip of a strong alcoholic beverage ca...


Contribuciones: MED-4994


,term,tf,df,length,tfidf_contribution,bm25_contribution
0,alcohol,4,166,137,0.107396,5.381987
1,benefit,5,314,137,0.083749,4.455824


... (27 outputs left out)

,rank,document_id,score,grade,length,title
0,1,MED-4668,9.046572,0,110,Cretinism revisited.


Contribuciones: MED-4668


,term,tf,df,length,tfidf_contribution,bm25_contribution
0,deaf,1,1,110,0.151697,9.046572


Menos de 10 candidatos; las posiciones restantes aportan cero.


## 11. Conclusiones
La comparación siguiente se genera a partir de las métricas de esta ejecución.
Los resultados apoyan una diferencia pequeña en el promedio global y diferencias
mayores por consulta; no se atribuye significación estadística sin un análisis
adicional. El corpus y la evaluación permanecen a nivel de documento original.
La principal limitación compartida es el desajuste de vocabulario y la ausencia
de comprensión semántica. Los scores son señales de recuperación, no conclusiones
médicas. Se entrega la implementación léxica solicitada, sin otros modelos.

In [10]:
tf, bm = summary.loc['tfidf'], summary.loc['bm25']
display(Markdown(f"TF-IDF: P@10={tf['Precision@10']:.6f}, R@10={tf['Recall@10']:.6f}, "
    f"NDCG@10={tf['NDCG@10']:.6f}. BM25: P@10={bm['Precision@10']:.6f}, "
    f"R@10={bm['Recall@10']:.6f}, NDCG@10={bm['NDCG@10']:.6f}. "
    f"Diferencia NDCG (BM25−TF-IDF): {bm['NDCG@10']-tf['NDCG@10']:.6f}."))

TF-IDF: P@10=0.232508, R@10=0.153092, NDCG@10=0.323104. BM25: P@10=0.238080, R@10=0.152893, NDCG@10=0.328409. Diferencia NDCG (BM25−TF-IDF): 0.005304.